# Lezione 3 — Modelli task-specific: sentiment, NER e classificazione

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccasadei-maggioli/corso-nlp-genai-2026/blob/main/lezione_3_pipelines_ner_sentiment/notebook_03_pipelines.ipynb)

Bentornato! 👋 Nella Lezione 1 abbiamo eseguito la prima pipeline di sentiment, nella
Lezione 2 abbiamo trasformato le recensioni in **embeddings** per la ricerca semantica.
Oggi facciamo un passo decisivo: usiamo **modelli specializzati** (piccoli e veloci) per
*estrarre informazioni strutturate* dalle recensioni in linguaggio libero.

In questa lezione:
1. capiamo la differenza tra **modelli task-specific** e **LLM generativi**, e quando
   conviene l'uno o l'altro;
2. analizziamo il **sentiment** delle recensioni (positivo/negativo);
3. estraiamo le **entità nominate** (NER): prodotti, marchi, luoghi…;
4. classifichiamo i **temi** delle recensioni con la tecnica **zero-shot** (senza
   addestrare nulla);
5. costruiamo un **DataFrame arricchito** che riusiamo nel progetto finale (Lezione 6).

> 🎯 **Filo conduttore:** continuiamo sulle stesse recensioni clienti in italiano.
> Oggi le trasformiamo da testo libero a **dati strutturati e interrogabili**.

---
### ⚙️ Prima di tutto: attiva la GPU T4
Menu **`Runtime` → `Change runtime type` → Hardware accelerator: `T4 GPU` → `Save`**.
I modelli di oggi sono piccoli, ma la GPU rende tutto molto più veloce.

## 1. Installiamo le librerie

Ci serve `transformers` (le pipeline di Hugging Face) e `sentencepiece` (il tokenizer
usato da alcuni dei modelli multilingua di oggi). `torch` e `pandas` sono già su Colab.

> 💡 Come ogni notebook del corso, questo installa da sé ciò che gli serve: puoi aprirlo
> in modo indipendente dalle altre lezioni.

In [ ]:
# -q = silenzioso. sentencepiece e protobuf servono al tokenizer di mDeBERTa.
!pip install -q "transformers>=4.45" "sentencepiece>=0.2" "protobuf>=4.0"
print("Librerie installate ✅")

## 2. Specializzati vs generativi: due strade per lo stesso problema 🧠

Per analizzare il testo abbiamo oggi **due famiglie** di modelli.

| | **Modelli task-specific** | **LLM generativi** |
|---|---|---|
| Dimensione | piccoli (decine/centinaia di MB) | grandi (GB) |
| Velocità | molto veloci | più lenti |
| Compito | **uno solo**, addestrato apposta | **tanti**, descritti a parole |
| Output | strutturato (etichette, span) | testo libero |
| Costo (hardware) | basso | alto |

- I **modelli task-specific** sono allenati per *un* compito (sentiment, NER,
  classificazione…). Sono piccoli, velocissimi e danno output già strutturato: perfetti
  per processare **migliaia** di recensioni in pochi secondi.
- Gli **LLM generativi** (li vedremo dalla Lezione 4) sono enormi e versatili: capiscono
  istruzioni libere e *generano* testo (riassunti, risposte). Più potenti ma più lenti e
  costosi.

> 👉 **Regola pratica:** se il compito è ben definito e si ripete su molti testi
> (es. "qual è il sentiment?"), un modello specializzato è la scelta migliore. Quando
> serve *generare*, *riassumere* o ragionare in modo aperto, allora servono gli LLM.

Lo strumento che usiamo per i modelli specializzati è la **`pipeline`** di Hugging Face:
sceglie il modello giusto, fa la tokenizzazione, l'inferenza e formatta l'output — tutto
in una riga. Oggi ne vediamo tre tipi: `text-classification`, `token-classification`
(NER) e `zero-shot-classification`.

## 3. Carichiamo il dataset di recensioni 🛒

Riprendiamo il dataset sintetico di recensioni in italiano (riproducibile, seed fisso).
Scarichiamo lo script generatore se non è già nella sessione e generiamo 200 recensioni.

In [ ]:
import os, torch
import pandas as pd

if not os.path.exists("genera_recensioni.py"):
    !wget -q https://raw.githubusercontent.com/ccasadei-maggioli/corso-nlp-genai-2026/main/dati/genera_recensioni.py

import genera_recensioni

df = pd.DataFrame(genera_recensioni.genera_recensioni(n=200, seed=42))

# device=0 -> GPU; -1 -> CPU. Lo passeremo a tutte le pipeline.
device = 0 if torch.cuda.is_available() else -1
print("Recensioni caricate:", len(df), "| device:", device)
df.head(3)

## 4. Task 1 — Sentiment: di che umore è la recensione? 😊😡

Iniziamo dal compito più semplice: classificare il **sentiment** in **negativo**,
**neutro** o **positivo**. Usiamo `neuraly/bert-base-italian-cased-sentiment`, un modello
addestrato apposta sull'italiano. È una pipeline di tipo `text-classification`.

In [ ]:
from transformers import pipeline

analisi_sentiment = pipeline(
    task="text-classification",
    model="neuraly/bert-base-italian-cased-sentiment",  # etichette: negative/neutral/positive
    device=device,
)

# Una prova al volo su due frasi.
print(analisi_sentiment("Prodotto fantastico, arrivato in un giorno!"))
print(analisi_sentiment("Pessima esperienza, non lo ricomprerò mai più."))

In [ ]:
# Applichiamolo a TUTTE le recensioni e aggiungiamo la colonna 'sentiment'.
testi = df["testo"].tolist()
predizioni = analisi_sentiment(testi, batch_size=16, truncation=True)

df["sentiment"] = [p["label"] for p in predizioni]

# Distribuzione delle classi (negative / neutral / positive).
print(df["sentiment"].value_counts())
df[["rating", "titolo", "sentiment", "testo"]].head(8)

## 5. Task 2 — NER: estrarre le entità dal testo 🏷️

La **NER** (*Named Entity Recognition*, riconoscimento di entità nominate) individua nel
testo i "nomi propri" e li classifica. Le categorie standard sono quattro:

- **PER** — persone (nomi di persona);
- **ORG** — organizzazioni, aziende, **marchi**;
- **LOC** — luoghi (città, paesi, regioni);
- **MISC** — altre entità nominate (prodotti, eventi, opere…).

Usiamo `Babelscape/wikineural-multilingual-ner`, un modello multilingua che supporta
bene l'italiano. È una pipeline `token-classification`. L'opzione
`aggregation_strategy="simple"` ricuce i token in **parole/entità intere** (altrimenti
otterremmo i singoli pezzi di parola).

In [ ]:
ner = pipeline(
    "token-classification",
    model="Babelscape/wikineural-multilingual-ner",
    aggregation_strategy="simple",
    device=device,
)

# Proviamo su una frase costruita ad hoc, con un marchio e una città.
esempio = "Ho comprato le Cuffie XSound Pro a Milano, spedizione velocissima."
for entita in ner(esempio):
    print(f"{entita['word']:25s} -> {entita['entity_group']:6s} (score {entita['score']:.2f})")

In [ ]:
# Applichiamo la NER ad alcune recensioni e mostriamo le entità trovate.
# (Le recensioni sintetiche contengono soprattutto nomi di prodotti/marchi -> ORG/MISC.)
for i in range(20, 25):
    testo = df.loc[i, "prodotto"]
    entita = ner(testo)
    etichette = [f"{e['word']} [{e['entity_group']}]" for e in entita]
    print(f"Recensione {df.loc[i, 'id']} ({df.loc[i, 'prodotto']}):")
    print("   Entità:", etichette if etichette else "(nessuna)")
    print()

In [ ]:
# Funzione di utilità: dato un testo, restituisce la lista delle entità come stringhe
# "parola [TIPO]". La useremo per arricchire il DataFrame.
def estrai_entita(testo):
    risultati = ner(testo)
    return [f"{e['word']} [{e['entity_group']}]" for e in risultati]

# Aggiungiamo la colonna 'entita' a tutto il DataFrame.
df["entita"] = df["prodotto"].apply(estrai_entita)
df[["prodotto", "entita"]][20:25]

## 6. Task 3 — Classificazione zero-shot: i temi della recensione 🎯

E se volessimo classificare le recensioni in **temi** (spedizione, prezzo, qualità…) ma
non abbiamo un modello addestrato proprio su *quelle* etichette?

Entra in gioco la **classificazione zero-shot** (*zero-shot* = "a zero esempi"):
classifichiamo il testo **senza alcun addestramento**, scegliendo le etichette **al volo**,
come semplici stringhe. Il modello (qui `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`, addestrato
sull'inferenza testuale multilingua) valuta quanto ogni etichetta è "implicata" dal testo.

> 💡 La forza dello zero-shot è la flessibilità: cambiare le etichette è gratis e immediato,
> non serve ri-addestrare nulla. È perfetto quando i temi possono cambiare nel tempo.

In [ ]:
zsc = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=device,
)

# Le etichette candidate: i temi che ci interessano nelle recensioni.
etichette = ["spedizione", "prezzo", "qualità del prodotto", "assistenza clienti", "facilità d'uso"]

# multi_label=True: ogni etichetta è valutata in modo indipendente (una recensione può
# parlare di più temi contemporaneamente).
esempio = df.loc[1, "testo"]
print("Testo:", esempio, "\n")

risultato = zsc(esempio, candidate_labels=etichette, multi_label=True)
for etichetta, punteggio in zip(risultato["labels"], risultato["scores"]):
    print(f"  {etichetta:22s} {punteggio:.2f}")

In [ ]:
# Classifichiamo TUTTE le recensioni e teniamo il TEMA DOMINANTE (etichetta con punteggio
# più alto, cioè l'argmax). Passiamo la lista di testi: la pipeline gestisce il batch.
risultati_temi = zsc(df["testo"].tolist(), candidate_labels=etichette, multi_label=True)

# Per ogni recensione, 'labels' è già ordinata per punteggio decrescente: labels[0] è
# il tema dominante (argmax).
df["tema_dominante"] = [r["labels"][0] for r in risultati_temi]

print("Distribuzione dei temi dominanti:")
print(df["tema_dominante"].value_counts())
df[["titolo", "tema_dominante", "testo"]].head(8)

## 7. Mettiamo tutto insieme: il DataFrame arricchito ✨

Abbiamo trasformato il testo libero in **tre nuove informazioni strutturate**:
`sentiment`, `entita` e `tema_dominante`. Componiamo una vista di arricchimento che le
mette in fila accanto a prodotto, categoria e rating.

In [ ]:
# Aggiungiamo una colonna sintetica con le sole entità "principali" (le prime 3).
df["entita_principali"] = df["entita"].apply(lambda lista: ", ".join(lista[:3]) if lista else "—")

arricchito = df[["id", "prodotto", "categoria", "rating", "sentiment", "tema_dominante", "entita_principali", "testo"]]
arricchito[20:25]

> 📦 **Da tenere a mente:** questo DataFrame arricchito — recensioni + sentiment + tema +
> entità — è esattamente il tipo di dato strutturato che **riuseremo nel progetto finale
> (Lezione 6)**. Lì lo useremo per filtrare, raggruppare e, soprattutto, alimentare il
> sistema di **RAG** che risponde a domande citando le recensioni. Estrarre struttura dal
> testo libero, come abbiamo fatto oggi, è il primo passo di quasi ogni applicazione NLP.

## 8. Esercizio 🏋️

La classificazione zero-shot rende facile **aggiungere una nuova etichetta** senza
ri-addestrare nulla. Aggiungiamo il tema **"tempi di consegna"** e ricalcoliamo il tema
dominante; poi guardiamo come si distribuiscono i temi **per rating**.

Completa la cella seguente dove vedi i `TODO`.

In [ ]:
# TODO 1: aggiungi "tempi di consegna" all'elenco delle etichette candidate.
etichette_estese = ["spedizione", "prezzo", "qualità del prodotto",
                    "assistenza clienti", "facilità d'uso"]  # <-- aggiungi qui la nuova etichetta

# TODO 2: riclassifica tutte le recensioni con le etichette estese (multi_label=True)
#         e salva il tema dominante nella colonna 'tema_esteso'.
risultati_estesi = None  # <-- sostituisci con la chiamata a zsc(...)
# df["tema_esteso"] = ...

# TODO 3: mostra la distribuzione dei temi per rating con una tabella di frequenze.
#         Suggerimento: pd.crosstab(df["rating"], df["tema_esteso"]).
# print(...)

### ✅ Soluzione

<details>
<summary>Mostra la soluzione</summary>

```python
import matplotlib.pyplot as plt
import seaborn as sns

# TODO 1: nuova etichetta aggiunta.
etichette_estese = ["spedizione", "prezzo", "qualità del prodotto",
                    "assistenza clienti", "facilità d'uso", "tempi di consegna"]

# TODO 2: riclassifica e salva il tema dominante.
risultati_estesi = zsc(df["testo"].tolist(), candidate_labels=etichette_estese, multi_label=True)
df["tema_esteso"] = [r["labels"][0] for r in risultati_estesi]

# TODO 3: distribuzione dei temi per rating.
cross_tab = pd.crosstab(df["rating"], df["tema_esteso"])
plt.figure(figsize=(10, 6))
sns.heatmap(cross_tab, annot=True, cmap="Reds", fmt="g")

plt.title('Distribuzione dei temi per rating')
plt.xlabel('Temi')
plt.ylabel('Rating')
plt.show()
```
</details>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Soluzione eseguibile (puoi confrontarla con il tuo tentativo) ---
etichette_estese = ["spedizione", "prezzo", "qualità del prodotto",
                    "assistenza clienti", "facilità d'uso", "tempi di consegna"]

risultati_estesi = zsc(df["testo"].tolist(), candidate_labels=etichette_estese, multi_label=True)
df["tema_esteso"] = [r["labels"][0] for r in risultati_estesi]

cross_tab = pd.crosstab(df["rating"], df["tema_esteso"])
plt.figure(figsize=(10, 6))
sns.heatmap(cross_tab, annot=True, cmap="Reds", fmt="g")

plt.title('Distribuzione dei temi per rating')
plt.xlabel('Temi')
plt.ylabel('Rating')
plt.show()

## 9. Riepilogo e prossimi passi ✅

Oggi abbiamo usato tre **modelli task-specific**, ognuno in una riga grazie alle pipeline:
- **Sentiment** (`text-classification`): positivo/negativo per ogni recensione;
- **NER** (`token-classification`): entità nominate (PER/ORG/LOC/MISC);
- **Zero-shot** (`zero-shot-classification`): temi scelti al volo, senza addestramento.

Abbiamo poi costruito un **DataFrame arricchito** (testo + sentiment + tema + entità) che
sarà il punto di partenza del **progetto finale**.

➡️ **Prossima lezione (Lezione 4):** i **modelli specializzati** sono perfetti quando il
compito è ben definito, ma cosa succede quando dobbiamo *riassumere* decine di recensioni,
*rispondere* a una domanda in linguaggio naturale o *generare* una risposta al cliente? Lì
i task non bastano più: servono gli **LLM generativi** open source, e impareremo a farli
girare sulla nostra GPU T4.

📦 Tutto il materiale del corso: https://github.com/ccasadei-maggioli/corso-nlp-genai-2026